In [2]:
from __future__ import annotations
import torch 
import sys
import os
import json
import math
import logging
from pathlib import Path
from typing import Callable, Optional
from uuid import uuid4
from collections import defaultdict
import types

In [3]:
import cv2
import numpy as np

from torchvision import models, transforms
from PIL import Image

In [4]:
import psycopg2
!pip install psycopg2-binary pgvector
from pgvector.psycopg2 import register_vector
from psycopg2.extras import execute_values
import kagglehub
import shutil


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\Users\fjm25\Desktop\Facundo\TUIA\CV\tp2\tuia-dog-recognition-app\tp2cv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
path_descarga = kagglehub.dataset_download("gpiosenka/70-dog-breedsimage-data-set")
print("Descargado en:", path_descarga)

PATH_DESTINO = r"C:\Users\fjm25\Desktop\Facundo\TUIA\CV\tp2\tuia-dog-recognition-app\data"


if os.path.exists(PATH_DESTINO):
    shutil.rmtree(PATH_DESTINO)


shutil.copytree(path_descarga, PATH_DESTINO)
print(f"Dataset copiado: {PATH_DESTINO}")

Descargado en: C:\Users\fjm25\.cache\kagglehub\datasets\gpiosenka\70-dog-breedsimage-data-set\versions\2
Dataset copiado: C:\Users\fjm25\Desktop\Facundo\TUIA\CV\tp2\tuia-dog-recognition-app\data


In [6]:
RUTA_SRC = r"C:\Users\fjm25\Desktop\Facundo\TUIA\CV\tp2\tuia-dog-recognition-app\src"

if RUTA_SRC not in sys.path:
    sys.path.insert(0, RUTA_SRC)
else:

    sys.path.remove(RUTA_SRC)
    sys.path.insert(0, RUTA_SRC)

try:
    from lib.schemas import EmbeddingRecord, Neighbor, SearchResult
    from lib.storage.base import EmbeddingStoreProtocol
    print("cargados")
except ModuleNotFoundError as e:
    print(f"Error{e}")

logger = logging.getLogger(__name__)

cargados


In [7]:
class SimilarityService:
    """Etapa 1: buscador de imagenes por similitud.

    Funciones a implementar por el estudiante:
      - extract_embedding(image)
      - search_similar_images(embedding, top_k)
      - predict_breed_from_neighbors(results)

    La orquestacion (search, index_image, persistencia y metricas de similitud)
    ya esta provista y no debe modificarse sin justificarlo en el informe.
    """

    def __init__(
        self,
        store: EmbeddingStoreProtocol,
        similarity_metric: str,
        similarity_threshold: float,
        top_k: int,
        image_size: int,
        model_name: str,
        url_resolver: Optional[Callable[[Path], Optional[str]]] = None,
    ) -> None:
        self.store = store
        self.similarity_metric = similarity_metric
        self.similarity_threshold = similarity_threshold
        self.top_k = top_k
        self.image_size = image_size
        self.model_name = model_name
        self.url_resolver = url_resolver

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.preprocess = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(self.image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        self.base_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.model = torch.nn.Sequential(*list(self.base_model.children())[:-1])
        self.model = self.model.to(self.device)
        self.model.eval()

In [8]:
def _load_image(self, source_path: str) -> np.ndarray:
        image = cv2.imread(str(source_path))
        if image is None:
            raise ValueError(f"Could not read image: {source_path}")
        # BGR uint8 (convencion OpenCV)
        return image

In [9]:
def extract_embedding(self, image: np.ndarray) -> list[float]:
      """
      Genera el embedding de una imagen usando un modelo pre-entrenado en
      ImageNet (ej: ResNet50, EfficientNet, ConvNeXt) sin la capa de
      clasificacion final.

      Sugerencias:
        - Preprocesar la imagen (resize a self.image_size, normalizacion ImageNet).
        - Usar torchvision.models o timm con pesos pre-entrenados.
        - Recordar que la imagen llega en BGR (OpenCV).
      Retorna una lista de floats de dimension EMBEDDING_DIM.
      """
  
      image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
      pil_img = Image.fromarray(image_rgb)
      tensor_img = self.preprocess(pil_img).unsqueeze(0).to(self.device)

      with torch.no_grad():
          output = self.model(tensor_img)              
          embedding_tensor = torch.flatten(output)     
          embedding_tensor = embedding_tensor.view(1, 1, -1)  
          embedding_tensor = torch.nn.functional.avg_pool1d(embedding_tensor, kernel_size=4, stride=4)
          embedding_tensor = embedding_tensor.view(-1).cpu()   
          embedding_list = embedding_tensor.numpy().tolist()

      return embedding_list

In [10]:
def search_similar_images(self, embedding: list[float], top_k: int) -> list[Neighbor]:
    """
    Recupera de la base vectorial las top_k imagenes mas similares.
    Sugerencias:
      - Con pgvector: self.store.search(embedding, top_k).
      - Con JSON: iterar self.store.all() y usar self.similarity(...).
      - Respetar SIMILARITY_METRIC (cosine | l2).
    Retorna una lista de Neighbor (path, breed, score) ordenada por score
    descendente.
    """

    embedding_numpy = np.array(embedding, dtype=np.float32)
    try:
        raw_results = self.store.search(embedding_numpy, top_k)
    except Exception:
        raw_results = self.store.search(embedding, top_k)

    if not raw_results:
        return []

    neighbors = []
    for item in raw_results:
        if isinstance(item, dict):
            path = item['path']
            breed = item['breed']
            ref_emb = item.get('embedding', [])
        else:
            path = getattr(item, 'path', '')
            breed = getattr(item, 'breed', '')
            ref_emb = getattr(item, 'embedding', [])

        score = self.similarity(embedding, ref_emb) if ref_emb else 0.0
        neighbors.append(Neighbor(path=path, breed=breed, score=score))

    neighbors.sort(key=lambda x: x.score, reverse=True)
    return neighbors

In [11]:
def predict_breed_from_neighbors(self, results: list[Neighbor]) -> tuple[str, float]:
        """
        Predice la raza a partir de los vecinos recuperados (ej: voto
        mayoritario, opcionalmente ponderado por score).

        Si el mejor score esta por debajo de self.similarity_threshold se
        considera "unknown". Retorna (raza, score).
        """

        if not results:
            return "unknown", 0.0

        best_score = results[0].score
        threshold = getattr(self, "similarity_threshold", 0.0)
        is_cosine = (getattr(self, "similarity_metric", "cosine") == "cosine")

        if is_cosine:
            if best_score < threshold:
                return "unknown", float(best_score)
            else:
                if best_score < threshold: 
                    return "unknown", float(best_score)

        breed_votes = defaultdict(float)
        
        for neighbor in results:
            if is_cosine:
                weight = neighbor.score
            else:
                weight = 1.0 / (neighbor.score + 1e-5)
                
            breed_votes[neighbor.breed] += weight


        predicted_breed = max(breed_votes, key=breed_votes.get)

        return predicted_breed, float(best_score)

In [ ]:
def cosine(self, a: np.ndarray, b: np.ndarray) -> float:
        denom = np.linalg.norm(a) * np.linalg.norm(b)
        if denom == 0:
            return 0.0
        return float(np.dot(a, b) / denom)

def l2_similarity(self, a: np.ndarray, b: np.ndarray) -> float:
        dist = float(np.linalg.norm(a - b))
        return 1.0 / (1.0 + dist)

def similarity(self, query: list[float], ref: list[float]) -> float:
        a = np.asarray(query, dtype=np.float32)
        b = np.asarray(ref, dtype=np.float32)
        if self.similarity_metric.lower() == "l2":
            return self._l2_similarity(a, b)
        return self._cosine(a, b)

In [ ]:
def index_image(
        self, image_path: str, breed: str, metadata: dict[str, object] | None = None
    ) -> EmbeddingRecord:
        """Extrae el embedding de una imagen del dataset y lo persiste en la base vectorial."""
        image = self._load_image(image_path)
        embedding = self.extract_embedding(image)
        record = EmbeddingRecord(
            id_imagen=str(uuid4()),
            embedding=embedding,
            path=str(image_path),
            breed=breed,
            metadata=metadata or {},
        )
        self.store.append(record)
        return record

def with_url(self, neighbor: Neighbor) -> Neighbor:
        if self.url_resolver is not None and not neighbor.url:
            neighbor.url = self.url_resolver(Path(neighbor.path))
        return neighbor

def search(
        self,
        source_path: str,
        output_path: Path,
        embedding_fn: Optional[Callable[[np.ndarray], list[float]]] = None,
        model_name: Optional[str] = None,
        top_k: Optional[int] = None,
    ) -> str:
        """Pipeline completo de la Etapa 1: embedding -> vecinos -> raza predicha."""
        image = self._load_image(source_path)
        extractor = embedding_fn or self.extract_embedding
        embedding = extractor(image)

        k = int(top_k) if top_k else self.top_k
        neighbors = [self._with_url(n) for n in self.search_similar_images(embedding, k)]
        breed, score = self.predict_breed_from_neighbors(neighbors)
        logger.info("Predicted breed: %s (score=%.4f) for %s", breed, score, source_path)

        payload = SearchResult(
            source_path=source_path,
            model=model_name or self.model_name,
            predicted_breed=breed,
            score=round(float(score), 4),
            neighbors=neighbors,
        )
        output_path.mkdir(parents=True, exist_ok=True)
        result_file = output_path / f"result-{uuid4()}.json"
        result_file.write_text(
            json.dumps(payload.model_dump(), ensure_ascii=True, indent=2),
            encoding="utf-8",
        )
        return str(result_file)

In [18]:

print("\nPostgreSQL y pgvector se compilaron e iniciaron")
from lib.config import settings
from lib.storage.pgvector_store import PgVectorEmbeddingStore
from lib.services.similarity_service import SimilarityService

print("Conectando al Postgres de Docker usando settings real...")
print("embedding_dim configurado:", settings.embedding_dim)
store = PgVectorEmbeddingStore(
    host=settings.postgres_host,
    port=settings.postgres_port,
    dbname=settings.postgres_db,
    user=settings.postgres_user,
    password=settings.postgres_password,
    embedding_dim=settings.embedding_dim,
)

service = SimilarityService(
    store=store,
    similarity_metric=settings.similarity_metric,
    similarity_threshold=settings.similarity_threshold,
    top_k=settings.top_k,
    image_size=settings.image_size,
    model_name=settings.embedding_model,
)
print("El store y el servicio se instanciaron correctamente")


PostgreSQL y pgvector se compilaron e iniciaron
Conectando al Postgres de Docker usando settings real...
embedding_dim configurado: 512
El store y el servicio se instanciaron correctamente


In [19]:
carpeta_imagenes = r"C:\Users\fjm25\Desktop\Facundo\TUIA\CV\tp2\tuia-dog-recognition-app\data\train"

image_records = []

if not os.path.exists(carpeta_imagenes):
    print(f"No se encontró la carpeta '{carpeta_imagenes}'")
else:
    for root, dirs, files in os.walk(carpeta_imagenes):
        for archivo in files:
            if archivo.lower().endswith(('.jpg', '.jpeg', '.png')):

                ruta_completa = os.path.join(root, archivo)
                raza = os.path.basename(root)
                image_records.append({"path": ruta_completa,"breed": raza})

    print(f"Se cargaron {len(image_records)} imágenes listas para indexar.")

Se cargaron 7946 imágenes listas para indexar.


In [21]:
with store.conn.cursor() as cur:
    cur.execute("TRUNCATE TABLE embeddings;")
print("tabla truncada")

tabla truncada


In [ ]:

contador_exitos = 0
print(f"Iniciando indexado de {len(image_records)} imágenes")

for idx, record in enumerate(image_records):
    try:
        service.index_image(image_path=record['path'], breed=record['breed'])
        contador_exitos += 1
        if (idx + 1) % 800 == 0:
            print(f"Procesadas e insertadas {idx + 1}/{len(image_records)} imágenes")
    except Exception as e:
        print(f"Error en índice {idx} ({record.get('path')}): {e}")
        continue 

print(f"\nFinalizado: {contador_exitos}/{len(image_records)} imágenes indexadas.")

Iniciando indexado de 7946 imágenes...
Procesadas e insertadas 800/7946 imágenes
Procesadas e insertadas 1600/7946 imágenes
Procesadas e insertadas 2400/7946 imágenes
Procesadas e insertadas 3200/7946 imágenes
Procesadas e insertadas 4000/7946 imágenes
Procesadas e insertadas 4800/7946 imágenes
Procesadas e insertadas 5600/7946 imágenes
Procesadas e insertadas 6400/7946 imágenes
Procesadas e insertadas 7200/7946 imágenes

Finalizado: 7946/7946 imágenes indexadas.


In [25]:
imagen_test_path = image_records[0]['path']
raza_real = image_records[0]['breed']
print(f"Imagen de consulta: {imagen_test_path} (Raza Real: {raza_real})\n")

img_bgr = cv2.imread(imagen_test_path)
if img_bgr is None:
    raise ValueError(f"No se pudo cargar la imagen en {imagen_test_path}")

embedding_query = service.extract_embedding(img_bgr)

vecinos = service.search_similar_images(embedding_query, top_k=10)
raza_predicha, score_obtenido = service.predict_breed_from_neighbors(vecinos)

print("RESULTADOS")
print(f"Raza Predicha por K-NN: {raza_predicha}")
print(f"Score/distancia mejor vecino: {score_obtenido}\n")

print(f"10 vecinos más cercanos en Postgres:")
for i, v in enumerate(vecinos):
    breed = v.breed if hasattr(v, 'breed') else v.get('breed')
    score = v.score if hasattr(v, 'score') else v.get('score', 0.0)
    path = v.path if hasattr(v, 'path') else v.get('path')
    print(f"  {i+1}️ Raza: {breed} | Score: {score:.4f} | Path: {path}")

Imagen de consulta: C:\Users\fjm25\Desktop\Facundo\TUIA\CV\tp2\tuia-dog-recognition-app\data\train\Afghan\001.jpg (Raza Real: Afghan)

RESULTADOS
Raza Predicha por K-NN: Afghan
Score/distancia mejor vecino: 1.0

10 vecinos más cercanos en Postgres:
  1️ Raza: Afghan | Score: 1.0000 | Path: C:\Users\fjm25\Desktop\Facundo\TUIA\CV\tp2\tuia-dog-recognition-app\data\train\Afghan\001.jpg
  2️ Raza: Afghan | Score: 1.0000 | Path: data/train/Afghan/001.jpg
  3️ Raza: Afghan | Score: 1.0000 | Path: data/train/Afghan/001.jpg
  4️ Raza: Afghan | Score: 0.9207 | Path: data/train/Afghan/048.jpg
  5️ Raza: Afghan | Score: 0.9207 | Path: C:\Users\fjm25\Desktop\Facundo\TUIA\CV\tp2\tuia-dog-recognition-app\data\train\Afghan\048.jpg
  6️ Raza: Afghan | Score: 0.9170 | Path: data/train/Afghan/024.jpg
  7️ Raza: Afghan | Score: 0.9170 | Path: C:\Users\fjm25\Desktop\Facundo\TUIA\CV\tp2\tuia-dog-recognition-app\data\train\Afghan\024.jpg
  8️ Raza: Afghan | Score: 0.9145 | Path: data/train/Afghan/062.jpg
  9

In [ ]:
from lib.evaluation.metrics import ndcg_at_k

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando el dispositivo: {device}")

if hasattr(service, 'model'):
    service.model.to(device)
    service.model.eval() 

correctas = 0
total = 0
suma_ndcg_10 = 0.0

print(f"Iniciando evaluación del modelo sobre {len(image_records)} registros")

with torch.no_grad():
    for idx, record in enumerate(image_records):
        try:
            ruta_str = record['path'].replace("\\", "/") 
            raza_real = record['breed']

            img_bgr = cv2.imread(ruta_str)
            if img_bgr is None:
                raise ValueError(f"No se pudo cargar la imagen en {ruta_str}")

            embedding_query = service.extract_embedding(img_bgr)

            k_vecinos = getattr(service, 'top_k', 10)
            vecinos = service.search_similar_images(embedding_query, top_k=k_vecinos)

            raza_predicha, _ = service.predict_breed_from_neighbors(vecinos)
            razas_vecinos = [v.breed if hasattr(v, 'breed') else v.get('breed') for v in vecinos[:10]]

            relevancias = [1 if raza == raza_real else 0 for raza in razas_vecinos]
            ndcg_query = ndcg_at_k(relevancias, k=10)

            suma_ndcg_10 += ndcg_query
            total += 1
            
            if raza_predicha == raza_real:
                correctas += 1

            if (idx + 1) % 800 == 0:
                accuracy_actual = (correctas / total) * 100
                ndcg_actual = (suma_ndcg_10 / total) * 100
                print(f"Evaluadas {idx + 1}/{len(image_records)} | Accuracy: {accuracy_actual:.2f}% | Mean NDCG@10: {ndcg_actual:.2f}%")

        except Exception as e:
            print(f"Error procesando índice {idx} ({record.get('path')}): {e}")
            continue

print("\n" + "="*40)
if total > 0:
    accuracy_final = (correctas / total) * 100
    ndcg_final = (suma_ndcg_10 / total) * 100
    print(f"Total de imágenes evaluadas con éxito: {total}")
    print(f"Predicciones de raza correctas: {correctas}")
    print(f"Accuracy Global: {accuracy_final:.2f}%")
    print(f"Mean NDCG@10 Global: {ndcg_final:.2f}%")
else:
    print("Error: No se pudo evaluar ningún registro de imagen.")

Usando el dispositivo: cpu
Iniciando evaluación del modelo sobre 7946 registros...
Evaluadas 800/7946 | Accuracy: 94.12% | Mean NDCG@10: 98.45%
Evaluadas 1600/7946 | Accuracy: 95.62% | Mean NDCG@10: 98.78%
Evaluadas 2400/7946 | Accuracy: 94.33% | Mean NDCG@10: 98.17%
Evaluadas 3200/7946 | Accuracy: 93.84% | Mean NDCG@10: 98.11%
Evaluadas 4000/7946 | Accuracy: 94.42% | Mean NDCG@10: 98.32%
Evaluadas 4800/7946 | Accuracy: 94.81% | Mean NDCG@10: 98.31%
Evaluadas 5600/7946 | Accuracy: 94.39% | Mean NDCG@10: 98.18%
Evaluadas 6400/7946 | Accuracy: 94.20% | Mean NDCG@10: 98.12%
Evaluadas 7200/7946 | Accuracy: 94.38% | Mean NDCG@10: 98.18%

Total de imágenes evaluadas con éxito: 7946
Predicciones de raza correctas: 7489
Accuracy Global: 94.25%
Mean NDCG@10 Global: 98.14%


========================== CÓDIGO PARA CORRER ENTERO EL PROGRAMA SIN HACERLO DESDE EL DOCKER ===========================

In [ ]:
from __future__ import annotations

import sys
import traceback
from pathlib import Path
import cv2
import torch
from lib.config import settings
from lib.storage.pgvector_store import PgVectorEmbeddingStore
from lib.services.similarity_service import SimilarityService
from lib.evaluation.metrics import ndcg_at_k


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for _ in range(5):
        if (current / "src" / "lib").exists():
            return current
        current = current.parent
    raise RuntimeError(
        "No se pudo encontrar la raíz del repo (se esperaba 'src/lib'). "
        "Verificá que estés corriendo el notebook desde dentro del repo clonado."
    )

REPO_ROOT = find_repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / "data" / "train"

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Raíz del repo detectada: {REPO_ROOT}")
print(f"Dataset esperado en: {DATA_DIR}")


print("Conectando al Postgres de Docker usando settings")
store = PgVectorEmbeddingStore(
    host=settings.postgres_host,
    port=settings.postgres_port,
    dbname=settings.postgres_db,
    user=settings.postgres_user,
    password=settings.postgres_password,
    embedding_dim=settings.embedding_dim,
)
print(f"embedding_dim configurado: {settings.embedding_dim}")

service = SimilarityService(
    store=store,
    similarity_metric=settings.similarity_metric,
    similarity_threshold=settings.similarity_threshold,
    top_k=settings.top_k,
    image_size=settings.image_size,
    model_name=settings.embedding_model,
)
print("Store y servicio instanciados")


if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"No se encontró '{DATA_DIR}'. Verificá que el dataset esté en "
        "data/train/<raza>/*.jpg en la raíz del repo (no se versiona en git, "
        "hay que copiarlo/descargarlo aparte según las instrucciones del README)."
    )

image_records = []
for breed_dir in sorted(p for p in DATA_DIR.iterdir() if p.is_dir()):
    breed = breed_dir.name
    for image_path in breed_dir.glob("*"):
        if image_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
            abs_path = image_path.resolve()
            rel_path = abs_path.relative_to(REPO_ROOT).as_posix()
            image_records.append({
                "path_abs": str(abs_path),
                "path_rel": rel_path,
                "breed": breed,
            })

print(f"Se cargaron {len(image_records)} imágenes listas para indexar.")


with store.conn.cursor() as cur:
    cur.execute("TRUNCATE TABLE embeddings;")
print("Base de datos vaciada para nueva indexación.\n")

print(f"Iniciando procesamiento e indexación para {len(image_records)} imágenes")
contador_exitos = 0
for idx, record in enumerate(image_records):
    try:
        service.index_image(image_path=record["path_rel"], breed=record["breed"])
        contador_exitos += 1

        if (idx + 1) % 800 == 0:
            print(f"Procesadas e insertadas {idx + 1}/{len(image_records)} imágenes")

    except Exception as e:
        print(f"Error en índice {idx} (Ruta: {record['path_rel']}): {e}")
        traceback.print_exc()
        continue

print("\n" + "=" * 40)
if contador_exitos == len(image_records):
    print(f"Las {len(image_records)} imágenes se guardaron con formato relativo")
else:
    print(f"Indexación finalizada. Se indexaron {contador_exitos} de {len(image_records)} imágenes.")
print("=" * 40)


imagen_test = image_records[0]
raza_real = imagen_test["breed"]
print(f"\nImagen de consulta: {imagen_test['path_rel']} (Raza Real: {raza_real})\n")

img_bgr = cv2.imread(imagen_test["path_abs"])
if img_bgr is None:
    raise ValueError(f"No se pudo cargar la imagen en {imagen_test['path_abs']}")

embedding_query = service.extract_embedding(img_bgr)
vecinos = service.search_similar_images(embedding_query, top_k=10)
raza_predicha, score_obtenido = service.predict_breed_from_neighbors(vecinos)

print("RESULTADOS")
print(f"Raza Predicha por K-NN: {raza_predicha}")
print(f"Score/distancia mejor vecino: {score_obtenido}\n")
print("10 vecinos más cercanos en Postgres:")
for i, v in enumerate(vecinos):
    breed = v.breed if hasattr(v, "breed") else v.get("breed")
    score = v.score if hasattr(v, "score") else v.get("score", 0.0)
    path = v.path if hasattr(v, "path") else v.get("path")
    print(f"  {i+1}️ Raza: {breed} | Score: {score:.4f} | Path: {path}")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsando el dispositivo: {device}")
if hasattr(service, "model"):
    service.model.to(device)
    service.model.eval()

correctas = 0
total = 0
suma_ndcg_10 = 0.0

print(f"Iniciando evaluación del modelo sobre {len(image_records)} registros...")
with torch.no_grad():
    for idx, record in enumerate(image_records):
        try:
            raza_real = record["breed"]
            img_bgr = cv2.imread(record["path_abs"])
            if img_bgr is None:
                raise ValueError(f"No se pudo cargar la imagen en {record['path_abs']}")
            
            embedding_query = service.extract_embedding(img_bgr)
            k_vecinos = getattr(service, "top_k", 10)
            vecinos = service.search_similar_images(embedding_query, top_k=k_vecinos)
            raza_predicha, _ = service.predict_breed_from_neighbors(vecinos)

            razas_vecinos = [v.breed if hasattr(v, "breed") else v.get("breed") for v in vecinos[:10]]
            relevancias = [1 if raza == raza_real else 0 for raza in razas_vecinos]
            ndcg_query = ndcg_at_k(relevancias, k=10)
            suma_ndcg_10 += ndcg_query
            total += 1

            if raza_predicha == raza_real:
                    correctas += 1

            if (idx + 1) % 800 == 0:
                    accuracy_actual = (correctas / total) * 100
                    ndcg_actual = (suma_ndcg_10 / total) * 100
                    print(f"Evaluadas {idx + 1}/{len(image_records)} | Accuracy: {accuracy_actual:.2f}% | Mean NDCG@10: {ndcg_actual:.2f}%")

        except Exception as e:
                print(f"Error procesando índice {idx} ({record.get('path_rel')}): {e}")
                continue
        
print("\n" + "=" * 40)
if total > 0:
    accuracy_final = (correctas / total) * 100
    ndcg_final = (suma_ndcg_10 / total) * 100
    print(f"Total de imágenes evaluadas con éxito: {total}")
    print(f"Predicciones de raza correctas: {correctas}")
    print(f"Accuracy Global: {accuracy_final:.2f}%")
    print(f"Mean NDCG@10 Global: {ndcg_final:.2f}%")
else:
    print("Error: No se pudo evaluar ningún registro de imagen.")